In [ ]:
import pandas as pd
from ollama import chat


In [8]:
MVRR_df = pd.read_csv('../assn1b/MVRR_data_Oct19_forclass.csv')
NPS_df = pd.read_csv('../assn1b/NPS_data_Oct19_forclass.csv')

# remove filler rows
MVRR_df = MVRR_df[(MVRR_df["trial_property"] == f"trial-MVRR") & (MVRR_df["type"] != "FILLER")]
NPS_df = NPS_df[(NPS_df["trial_property"] == f"trial-NPS") & (NPS_df["type"] != "FILLER")]


In [14]:
fragments_NPS = NPS_df['sentence_fragment'].unique()

In [15]:
fragments_MVRR = MVRR_df['sentence_fragment'].unique()

In [ ]:
prompt = f"""
Continue this sentence with a natural continuation. Output only the continuation, no explanation. Sentence: {fragment}
"""



response = chat(
    model="qwen3:8b",
    messages=[
        {
            "role": "user",
            "content": (prompt
            )
        }
    ],
    think=False,
    options={
        "temperature": 1.0
    }
)

response.message.content

the injured runner treated by the paramedics was quickly stabilized and transported to the nearest hospital for further care.


In [33]:
model = "qwen3:8b"
n_runs = 20


MVRR_results = []

for item_number, fragment in enumerate(fragments_MVRR, start=1):
    for run in range(1, n_runs + 1):
        prompt = f"""
        Continue this sentence naturally. DO NOT EXPLAIN YOUR ANSWER. Only continue the sentence.
        Sentence: {fragment}
        """

        response = chat(
            model=model,
            messages=[
                {"role": "user", "content": prompt}
            ],
            think=False,
            options={
                "temperature": 1.0,
                "num_predict": 40
            }
        )

        completion = response.message.content.strip()

        MVRR_results.append({
            "run_number": run,
            "item_number": item_number,
            "fragment": fragment,
            "completion": completion,
            "full_sentence": f"{fragment} {completion}"
        })

# Convert to DataFrame
llm_MVRR_results = pd.DataFrame(MVRR_results)

In [36]:
llm_MVRR_results

,run_number,item_number,fragment,completion,full_sentence
0,1,1,The passionate activist arrested,for protesting against the government's enviro...,The passionate activist arrested for protestin...
1,2,1,The passionate activist arrested,The passionate activist arrested for organizin...,The passionate activist arrested The passionat...
2,3,1,The passionate activist arrested,for protesting against the government's enviro...,The passionate activist arrested for protestin...
3,4,1,The passionate activist arrested,for protesting against the corrupt government ...,The passionate activist arrested for protestin...
4,5,1,The passionate activist arrested,The passionate activist arrested for organizin...,The passionate activist arrested The passionat...
...,...,...,...,...,...
395,16,20,The exciting study reviewed,The exciting study reviewed the long-term effe...,The exciting study reviewed The exciting study...
396,17,20,The exciting study reviewed,The exciting study reviewed a wide range of hi...,The exciting study reviewed The exciting study...
397,18,20,The exciting study reviewed,The exciting study reviewed a wide range of hi...,The exciting study reviewed The exciting study...
398,19,20,The exciting study reviewed,The exciting study reviewed the long-term effe...,The exciting study reviewed The exciting study...


In [37]:
llm_MVRR_results[['run_number','fragment','completion']].to_csv("llm_MVRR_results.csv",index=False)

In [38]:
model = "qwen3:8b"
n_runs = 20

NPS_results = []

for item_number, fragment in enumerate(fragments_NPS, start=1):
    for run in range(1, n_runs + 1):

        prompt = f"""
        Continue this sentence naturally. DO NOT EXPLAIN YOUR ANSWER. Only continue the sentence.
        Sentence: {fragment}
        """

        response = chat(
            model=model,
            messages=[
                {"role": "user", "content": prompt}
            ],
            think=False,
            options={
                "temperature": 1.0,
                "num_predict": 40
            }
        )

        completion = response.message.content.strip()

        NPS_results.append({
            "run_number": run, 
            "item_number": item_number,
            "fragment": fragment,
            "completion": completion,
            "full_sentence": f"{fragment} {completion}"
        })

# Convert to DataFrame
llm_NPS_results = pd.DataFrame(NPS_results)


In [40]:
llm_NPS_results[['run_number','fragment','completion']].to_csv("llm_NPS_results.csv",index=False)